In [1]:
import pandas as pd

In [14]:
df = pd.read_parquet(r"D:\pypipeline\data\processed\hourly\us_paro_hourly\train_processed.parquet")

In [15]:
df

,pm25,o3,pm25_target,o3_target,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,...,pm25_plus_20h,o3_plus_20h,pm25_plus_21h,o3_plus_21h,pm25_plus_22h,o3_plus_22h,pm25_plus_23h,o3_plus_23h,pm25_plus_24h,o3_plus_24h
date,,,,,,,,,,,,,,,,,,,,,
2017-03-03 05:00:00+05:45,54.5,0.004,54.5,0.004,5,4,0,1,0,0,...,153.5,0.009,107.2,0.007,80.6,0.007,80.6,0.007,-999.0,-0.999
2017-03-03 06:00:00+05:45,73.3,0.004,73.3,0.004,6,4,0,0,0,0,...,107.2,0.007,80.6,0.007,80.6,0.007,-999.0,-0.999,-999.0,-0.999
2017-03-03 07:00:00+05:45,76.1,0.003,76.1,0.003,7,4,0,0,0,0,...,80.6,0.007,80.6,0.007,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999
2017-03-03 08:00:00+05:45,95.0,0.005,95.0,0.005,8,4,0,0,0,0,...,80.6,0.007,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999
2017-03-03 09:00:00+05:45,208.5,0.009,208.5,0.009,9,4,0,0,0,0,...,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999,-999.0,-0.999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-27 14:00:00+05:45,NaN,0.044,-999.0,NaN,14,4,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-12-27 15:00:00+05:45,NaN,0.044,-999.0,NaN,15,4,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-12-27 16:00:00+05:45,NaN,0.044,-999.0,NaN,16,4,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
from preprocess import HORIZON
from train import TARGET_COL, VALUE_MIN,VALUE_MAX


target_key = f"{TARGET_COL}_plus_1h"

features_exclude = [f"{TARGET_COL}_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"o3_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"pm25_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       ["segment_id", "imputation_confidence"]

test_mask = df[target_key].notnull() & (df[target_key] >= VALUE_MIN) \
            & (df[target_key] <= VALUE_MAX)

y = df.loc[test_mask, target_key]
            

df_test_h = df[test_mask].drop(features_exclude, axis=1).copy()
df_test_eval = df_test_h.join(y)

    

In [17]:
df_test_eval

,pm25,o3,pm25_target,o3_target,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,...,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio,pm25_plus_1h
date,,,,,,,,,,,,,,,,,,,,,
2017-03-03 05:00:00+05:45,54.5,0.004,54.5,0.004,5,4,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13625.000000,73.3
2017-03-03 06:00:00+05:45,73.3,0.004,73.3,0.004,6,4,0,0,0,0,...,54.500000,0.004000,54.500000,0.004000,NaN,NaN,NaN,NaN,18325.000000,76.1
2017-03-03 07:00:00+05:45,76.1,0.003,76.1,0.003,7,4,0,0,0,0,...,63.900000,0.004000,63.900000,0.004000,10.80,-5.000000e-04,10.800000,-0.000500,25366.666667,95.0
2017-03-03 08:00:00+05:45,95.0,0.005,95.0,0.005,8,4,0,0,0,0,...,67.966667,0.003667,67.966667,0.003667,10.85,5.000000e-04,12.430000,0.000200,19000.000000,208.5
2017-03-03 09:00:00+05:45,208.5,0.009,208.5,0.009,9,4,0,0,0,0,...,74.725000,0.004000,74.725000,0.004000,66.20,3.000000e-03,32.970000,0.001100,23166.666667,191.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-26 07:00:00+05:45,88.0,0.002,88.0,0.002,7,3,0,0,0,0,...,90.000000,0.004750,83.500000,0.018833,13.50,6.756970e-20,-6.961538,-0.000615,44000.000000,101.0
2019-12-26 08:00:00+05:45,101.0,0.002,101.0,0.002,8,3,0,0,0,0,...,91.083333,0.003667,84.541667,0.018833,17.00,6.756970e-20,-7.038462,-0.000346,50500.000000,128.0
2019-12-26 09:00:00+05:45,128.0,0.006,128.0,0.006,9,3,0,0,0,0,...,92.750000,0.002917,85.250000,0.018750,20.00,2.000000e-03,-3.034965,0.000003,21333.333333,116.0


0       2017-03-03 05:00:00+05:45
1       2017-03-03 06:00:00+05:45
2       2017-03-03 07:00:00+05:45
3       2017-03-03 08:00:00+05:45
4       2017-03-03 09:00:00+05:45
                   ...           
24705   2019-12-27 14:00:00+05:45
24706   2019-12-27 15:00:00+05:45
24707   2019-12-27 16:00:00+05:45
24708   2019-12-27 17:00:00+05:45
24709   2019-12-27 18:00:00+05:45
Name: date, Length: 24710, dtype: datetime64[ns, pytz.FixedOffset(345)]